In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### Utility functions

import os
import sys
import importlib.util

project_utils = os.path.join(os.getcwd(), '..', '..')
transformation_path = os.path.join(project_utils, 'utils', 'transformation.py')

spec = importlib.util.spec_from_file_location("transformation", transformation_path)
transformation = importlib.util.module_from_spec(spec)
spec.loader.exec_module(transformation)

reusable = transformation.reusable

### DimUser - Auto Loader Streaming Pipeline

In [0]:
df_user = spark.readStream.format("cloudFiles")\
          .option("cloudFiles.format", "parquet")\
          .option("cloudFiles.schemaLocation", "abfss://silver@swap01storageaccount.dfs.core.windows.net/DimUser/checkpoint")\
          .load("abfss://bronze@swap01storageaccount.dfs.core.windows.net/DIMUSER")


In [0]:
# Apply transformations
df_user_obj = reusable()
df_user = df_user_obj.dropColumns(df_user, ['_rescued_data'])

# Deduplicate by user_id
df_user = df_user.dropDuplicates(['user_id'])

# Preview the data
display(df_user, checkpointLocation="abfss://silver@swap01storageaccount.dfs.core.windows.net/DimUser/preview_checkpoint")

In [0]:
df_user.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@swap01storageaccount.dfs.core.windows.net/DimUser/checkpoint")\
        .trigger(once=True)\
            .option("path", "abfss://silver@swap01storageaccount.dfs.core.windows.net/DimUser/data")\
            .toTable("`spotify-catalog`.silver.DimUser")

### DimArtist

In [0]:
df_artist = spark.readStream.format("cloudFiles")\
          .option("cloudFiles.format", "parquet")\
          .option("cloudFiles.schemaLocation", "abfss://silver@swap01storageaccount.dfs.core.windows.net/DimArtist/checkpoint")\
          .load("abfss://bronze@swap01storageaccount.dfs.core.windows.net/DIMARTIST")

In [0]:
# Apply transformations
df_artist_obj = reusable()
df_artist = df_artist_obj.dropColumns(df_artist, ['_rescued_data'])

# Deduplicate by artist_id
df_artist = df_artist.dropDuplicates(['artist_id'])

# Preview the data
display(df_artist, checkpointLocation="abfss://silver@swap01storageaccount.dfs.core.windows.net/DimArtist/preview_checkpoint")

In [0]:
df_artist.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@swap01storageaccount.dfs.core.windows.net/DimArtist/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@swap01storageaccount.dfs.core.windows.net/DimArtist/data")\
    .toTable("`spotify-catalog`.silver.DimArtist")

### DimDate

In [0]:
df_date = spark.readStream.format("cloudFiles")\
          .option("cloudFiles.format", "parquet")\
          .option("cloudFiles.schemaLocation", "abfss://silver@swap01storageaccount.dfs.core.windows.net/DimDate/checkpoint")\
          .load("abfss://bronze@swap01storageaccount.dfs.core.windows.net/DIMDATE")

In [0]:
# Apply transformations
df_date_obj = reusable()
df_date = df_date_obj.dropColumns(df_date, ['_rescued_data'])

# Preview the data
display(df_date, checkpointLocation="abfss://silver@swap01storageaccount.dfs.core.windows.net/DimDate/preview_checkpoint")

In [0]:
df_date.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@swap01storageaccount.dfs.core.windows.net/DimDate/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@swap01storageaccount.dfs.core.windows.net/DimDate/data")\
    .toTable("`spotify-catalog`.silver.DimDate")

### DimTrack

In [0]:
df_track = spark.readStream.format("cloudFiles")\
          .option("cloudFiles.format", "parquet")\
          .option("cloudFiles.schemaLocation", "abfss://silver@swap01storageaccount.dfs.core.windows.net/DimTrack/checkpoint")\
          .load("abfss://bronze@swap01storageaccount.dfs.core.windows.net/DIMTRACK")

In [0]:
# Apply transformations
df_track_obj = reusable()
df_track = df_track_obj.dropColumns(df_track, ['_rescued_data'])

# Deduplicate by track_id
df_track = df_track.dropDuplicates(['track_id'])

df_track = df_track.withColumn("durationFlag" , when(col('duration_sec')<150, "low").when(col('duration_sec')<300, "medium").otherwise("high"))
                                               
                                                               
df_track = df_track.withColumn("track_name" , regexp_replace(col('track_name'), '-', '  '))                                                           

# Preview the data
display(df_track, checkpointLocation="abfss://silver@swap01storageaccount.dfs.core.windows.net/DimTrack/preview_checkpoint")

In [0]:
df_track.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@swap01storageaccount.dfs.core.windows.net/DimTrack/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@swap01storageaccount.dfs.core.windows.net/DimTrack/data")\
    .toTable("`spotify-catalog`.silver.DimTrack")

### FactStream

In [0]:
df_factstream = spark.readStream.format("cloudFiles")\
          .option("cloudFiles.format", "parquet")\
          .option("cloudFiles.schemaLocation", "abfss://silver@swap01storageaccount.dfs.core.windows.net/FactStream/checkpoint")\
          .load("abfss://bronze@swap01storageaccount.dfs.core.windows.net/FACTSTREAM")

In [0]:
# Apply transformations
df_factstream_obj = reusable()
df_factstream = df_factstream_obj.dropColumns(df_factstream, ['_rescued_data'])

# Preview the data
display(df_factstream, checkpointLocation="abfss://silver@swap01storageaccount.dfs.core.windows.net/FactStream/preview_checkpoint")

In [0]:
df_factstream.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@swap01storageaccount.dfs.core.windows.net/FactStream/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@swap01storageaccount.dfs.core.windows.net/FactStream/data")\
    .toTable("`spotify-catalog`.silver.FactStream")